In [ ]:
# Thermal 3D Vision Overview Notebook

import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
from tqdm import tqdm

# Add project root to path to import modules
sys.path.append("..")
from data.freiburg_dataset import FreiburgThermalDataset
from models.dust3r import DUSt3R
from models.mast3r import MASt3R
from utils.visualization import  visualize_pointcloud
from utils.metrics import compute_depth_metrics

# Configuration
CONFIG_PATH = "../config/model_config.yaml"
CHECKPOINT_PATH = "../checkpoints/latest.pth"
RESULTS_DIR = "../results"

# Load config
import yaml
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# Device
device = torch.device(config['device'] if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# %% [markdown]
# ## 1. Data Exploration

# %%
# Create dataset
dataset = FreiburgThermalDataset(
    root_dir=config['data']['train_path'],
    transform=None  # No transforms for visualization
)

# %% [markdown]
# ### 1.1 Dataset Overview

# %%
print(f"Dataset size: {len(dataset)}")
print(f"Number of sequences: {len(dataset.sequences)}")

# Show a few sample thermal images
plt.figure(figsize=(15, 10))
for i in range(3):
    sample = dataset[i]
    thermal_img = sample['img1'].numpy()
    
    # Normalize for visualization
    thermal_img = (thermal_img - thermal_img.min()) / (thermal_img.max() - thermal_img.min())
    
    plt.subplot(1, 3, i+1)
    plt.imshow(thermal_img.transpose(1, 2, 0))
    plt.title(f"Sample {i}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 1.2 Camera Intrinsics

# %%
# Display camera intrinsics
intrinsics = dataset.intrinsics
print("Camera Intrinsics:")
print(intrinsics)

print(f"Focal length (fx, fy): ({intrinsics[0, 0]}, {intrinsics[1, 1]})")
print(f"Principal point (cx, cy): ({intrinsics[0, 2]}, {intrinsics[1, 2]})")

# %% [markdown]
# ## 2. Model Architecture

# %%
# Load models
dust3r_model = DUSt3R(backbone=config['model']['backbone'], pretrained=True)
mast3r_model = MASt3R(pretrained=True)

# Display model architectures
print("DUSt3R Model Architecture:")
print(dust3r_model)

print("\nMASt3R Model Architecture:")
print(mast3r_model)

# %% [markdown]
# ## 3. Pseudo-GT Generation

# %%
# Load a sample pair of images
sample = dataset[0]
img1 = sample['img1'].unsqueeze(0).to(device)
img2 = sample['img2'].unsqueeze(0).to(device)

# Generate pseudo-GT with MASt3R
mast3r_model = mast3r_model.to(device)
mast3r_model.eval()

with torch.no_grad():
    outputs = mast3r_model(img1, img2)
    pointmap1 = outputs['pointmap1']
    pointmap2 = outputs['pointmap2']
    
    # Extract depth from pointmaps
    depth1 = pointmap1[0, 2].cpu().numpy()

# Visualize the pseudo-GT depth
plt.figure(figsize=(10, 8))
plt.imshow(depth1, cmap='viridis')
plt.colorbar(label='Depth')
plt.title('Pseudo-GT Depth from MASt3R')
plt.show()

# %% [markdown]
# ## 4. Model Training and Fine-tuning

# %%
# Load the fine-tuned model
model = DUSt3R(backbone=config['model']['backbone'], pretrained=False)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model'])
model = model.to(device)
model.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")

# %% [markdown]
# ## 5. Qualitative Results

# %%
# Load test dataset
test_dataset = FreiburgThermalDataset(
    root_dir=config['data']['test_path'],
    transform=None  # No transforms for visualization
)

# Select a few samples to visualize
sample_indices = [0, 10, 20, 30, 40]

plt.figure(figsize=(15, 12))
for i, idx in enumerate(sample_indices):
    sample = test_dataset[idx]
    img = sample['img1'].unsqueeze(0).to(device)
    
    # Forward pass through model
    with torch.no_grad():
        outputs = model(img, img)  # Same image for both inputs (we only care about depth)
        pred_pointmap = outputs['pointmap1'][0].cpu().numpy()
        pred_depth = pred_pointmap[2]  # Z component
    
    # Normalize thermal image for visualization
    thermal_img = sample['img1'].numpy().transpose(1, 2, 0)
    thermal_img = (thermal_img - thermal_img.min()) / (thermal_img.max() - thermal_img.min())
    
    # Plot thermal image
    plt.subplot(len(sample_indices), 2, i*2+1)
    plt.imshow(thermal_img)
    plt.title(f"Sample {idx} - Thermal")
    plt.axis('off')
    
    # Plot predicted depth
    plt.subplot(len(sample_indices), 2, i*2+2)
    plt.imshow(pred_depth, cmap='viridis')
    plt.title(f"Sample {idx} - Predicted Depth")
    plt.axis('off')

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 6. Quantitative Evaluation

# %%
# Load evaluation results
metrics_path = os.path.join(RESULTS_DIR, "freiburg_evaluation", "metrics.txt")

if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = {}
        for line in f:
            key, value = line.strip().split(': ')
            metrics[key] = float(value)
    
    # Display metrics
    print("Evaluation Metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    
    # Plot metrics as a bar chart
    plt.figure(figsize=(10, 6))
    plt.bar(metrics.keys(), metrics.values())
    plt.title('Evaluation Metrics')
    plt.ylabel('Value')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print(f"Metrics file not found at {metrics_path}")

# %% [markdown]
# ## 7. Comparison with Baseline Models

# %%
# Compare our fine-tuned model with base DUSt3R and MASt3R
models = {
    'Fine-tuned DUSt3R': model,
    'Base DUSt3R': DUSt3R(backbone=config['model']['backbone'], pretrained=True).to(device),
    'MASt3R': MASt3R(pretrained=True).to(device)
}

sample = test_dataset[15]  # Choose a sample
img1 = sample['img1'].unsqueeze(0).to(device)
img2 = sample['img2'].unsqueeze(0).to(device)

plt.figure(figsize=(15, 8))

# Show thermal image
thermal_img = sample['img1'].numpy().transpose(1, 2, 0)
thermal_img = (thermal_img - thermal_img.min()) / (thermal_img.max() - thermal_img.min())
plt.subplot(1, 4, 1)
plt.imshow(thermal_img)
plt.title("Thermal Image")
plt.axis('off')

# Show depth from each model
for i, (name, model) in enumerate(models.items()):
    model.eval()
    with torch.no_grad():
        outputs = model(img1, img2)
        pred_depth = outputs['pointmap1'][0, 2].cpu().numpy()
    
    plt.subplot(1, 4, i+2)
    plt.imshow(pred_depth, cmap='viridis')
    plt.title(f"{name}")
    plt.axis('off')

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 8. AIS Data Results

# %%
# Show a few results from AIS test data
ais_results_dir = os.path.join(RESULTS_DIR, "ais_evaluation")

if os.path.exists(ais_results_dir):
    sample_dirs = [d for d in os.listdir(ais_results_dir) if d.startswith("sample_")][:3]
    
    plt.figure(figsize=(15, 10))
    for i, sample_dir in enumerate(sample_dirs):
        # Thermal image
        thermal_path = os.path.join(ais_results_dir, sample_dir, "thermal.png")
        thermal_img = plt.imread(thermal_path)
        
        # Predicted depth
        depth_path = os.path.join(ais_results_dir, sample_dir, "depth_pred.png")
        depth_img = plt.imread(depth_path)
        
        # Plot
        plt.subplot(len(sample_dirs), 2, i*2+1)
        plt.imshow(thermal_img)
        plt.title(f"AIS Sample {i} - Thermal")
        plt.axis('off')
        
        plt.subplot(len(sample_dirs), 2, i*2+2)
        plt.imshow(depth_img)
        plt.title(f"AIS Sample {i} - Depth")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"AIS results directory not found at {ais_results_dir}")

# %% [markdown]
# ## 9. Conclusion and Next Steps

# %%
# Summary of findings
print("Summary of Project Findings:")
print("1. Successfully fine-tuned DUSt3R model on thermal images")
print("2. Achieved depth estimation accuracy of X on Freiburg dataset")
print("3. Model generalizes well to AIS test data")
print("4. Key challenges included ...")

# Next steps
print("\nPotential Next Steps:")
print("1. Experiment with larger backbone models")
print("2. Incorporate additional thermal datasets")
print("3. Test on more diverse environmental conditions")
print("4. Integrate with downstream robotics tasks")